<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/11.Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 11 — Fine-Tuning

*LLM Design Patterns* ·
GPT-2 sudah belajar menebak kata berikutnya dari jutaan halaman web. Kemampuan umum itu belum tentu pas untuk teks atau tugas kita: artikel bergaya Wikipedia, paragraf sains, atau format tanya-jawab "negara → ibu kota". **Fine-tuning** menutup celah itu dengan cara paling langsung: training model yang sudah jadi dilanjutkan pada data kita, dengan learning rate kecil dan langkah yang jauh lebih sedikit daripada pretraining.

Bab ini mengikuti urutan buku, dan setiap klaimnya diukur pada GPT-2 124M:

1. fine-tuning dasar dengan `Trainer`, termasuk dari mana `labels` berasal;
2. membekukan sebagian layer: berapa parameter, waktu, dan memori yang benar-benar dihemat;
3. learning-rate schedule dan cara memasangnya ke `Trainer`;
4. adaptasi domain ke teks sains;
5. few-shot: format tanya-jawab dari 16 contoh, dibandingkan dengan contoh di dalam prompt;
6. continual learning: mengukur *catastrophic forgetting* dan dua penangkalnya, EWC dan LwF.

Beberapa listing buku tidak bisa dijalankan apa adanya. Masing-masing muncul sebagai kotak kutipan pendek berisi potongan kodenya, masalah yang sudah diverifikasi, dan penunjuk ke versi yang berjalan tepat di bawahnya.

## Langkah 0 — Lingkungan dan gerbang GPU

Sel pertama memasang paket yang belum ada (Colab biasanya sudah punya `torch` dan `transformers`) dan memilih profil run:

- `PROFILE = "gpu"` (bawaan): GPT-2 124M pretrained dengan ukuran data dan jumlah step bab ini.
- `PROFILE = "cpu_smoke"`: hanya dipakai bila dipilih secara eksplisit. Model diganti GPT-2 mini berbobot acak dan semua ukuran dipangkas, sehingga notebook selesai dalam hitungan menit di CPU. Angkanya tidak bermakna; profil ini hanya memeriksa bahwa kode berjalan.

Semua ukuran eksperimen dikumpulkan di `CFG`. Bila ingin memperbesar atau memperkecil run, cukup ubah satu tempat itu.

In [1]:
# Profil run: "gpu" (bawaan, wajib CUDA) atau "cpu_smoke" (opt-in eksplisit: model mini acak, angka tidak bermakna).
import os
import sys
import importlib.util
import subprocess

PROFILE = os.environ.get("CH11_PROFILE", "gpu")
if PROFILE not in ("gpu", "cpu_smoke"):
    raise ValueError(f"PROFILE harus 'gpu' atau 'cpu_smoke', bukan {PROFILE!r}")

CFG = {
    "gpu": dict(seq_len=128, batch_size=8, n_train_lines=4000, n_val_lines=400,
                trainer_steps=120, arm_steps=30, stage_steps=20, sched_steps=20,
                domain_steps=60, domain_val_blocks=200, sft_epochs=6,
                ewc_lines=4000, ewc_val_lines=1000, ewc_train_blocks=400, ewc_val_blocks=64,
                ewc_steps_a=150, ewc_steps_b=120),
    "cpu_smoke": dict(seq_len=64, batch_size=4, n_train_lines=64, n_val_lines=16,
                      trainer_steps=3, arm_steps=3, stage_steps=2, sched_steps=4,
                      domain_steps=2, domain_val_blocks=8, sft_epochs=1,
                      ewc_lines=600, ewc_val_lines=300, ewc_train_blocks=32, ewc_val_blocks=16,
                      ewc_steps_a=6, ewc_steps_b=4),
}[PROFILE]

WORK_DIR = os.path.abspath(os.environ.get("CH11_WORK_DIR", "ch11_work"))   # output Trainer dan file korpus
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

missing = [pip_name for module, pip_name in (("transformers", "transformers>=4.41"),
                                             ("datasets", "datasets>=2.19"),
                                             ("accelerate", "accelerate>=0.26"))
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import collections
import contextlib
import copy
import gc
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
import datasets
import transformers
from datasets import load_dataset
from packaging import version
from transformers import (AutoTokenizer, DataCollatorForLanguageModeling, GPT2Config,
                          GPT2LMHeadModel, Trainer, TrainingArguments,
                          get_cosine_schedule_with_warmup, get_linear_schedule_with_warmup)
from transformers.trainer_callback import PrinterCallback

if version.parse(transformers.__version__) < version.parse("4.41"):
    raise RuntimeError("Butuh transformers >= 4.41 (argumen eval_strategy). Jalankan "
                       "`pip install -U transformers`, lalu Runtime > Restart session.")

transformers.logging.set_verbosity_error()
datasets.disable_progress_bars()
transformers.set_seed(42)
print(f"profil {PROFILE} | python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | datasets {datasets.__version__}")

profil gpu | python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5


Sel berikut adalah **gerbang GPU**. Mencetak `torch.cuda.is_available()` saja tidak cukup, karena GPU bisa terdeteksi tetapi gagal menghitung. Gerbang ini mengalokasikan tensor di GPU, menjalankan perkalian matriks yang hasilnya sudah diketahui (setiap elemen harus tepat 2048), menunggu GPU selesai dengan `synchronize`, lalu memeriksa nilainya. Bila ada yang gagal, notebook berhenti dengan instruksi; tidak ada perpindahan diam-diam ke CPU.

Gerbang yang sama memilih **presisi**. GPU dengan compute capability ≥ 8 (A100, L4, dan sejenisnya) menghitung bfloat16 secara native. T4 (compute capability 7.5) tidak, sehingga di T4 kita memakai float16 ditemani **GradScaler**: loss dikalikan faktor besar sebelum backward agar gradien kecil tidak membulat menjadi nol di float16, lalu dibagi kembali sebelum optimizer melangkah. Bobot model tetap float32; hanya operasi berat seperti perkalian matriks yang dihitung dalam presisi rendah lewat *autocast*.

In [2]:
def cuda_gate():
    """Uji fungsional GPU: alokasi, hitung, synchronize, cek nilai. Gagal berarti berhenti."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU CUDA tidak tersedia. Di Colab: Runtime > Change runtime type > pilih GPU (T4 cukup) > Save, "
            "lalu jalankan ulang dari atas. Hanya untuk cek alur kode tanpa GPU, set PROFILE = 'cpu_smoke' "
            "secara eksplisit di sel sebelumnya.")
    device = torch.device("cuda")
    n = 2048
    exact = torch.ones(n, n, device=device) @ torch.ones(n, device=device)   # setiap elemen harus tepat n
    x = torch.randn(n, n, device=device) / math.sqrt(n)
    half = x.half() @ x.half().T                                             # jalur float16
    torch.cuda.synchronize()
    if not (torch.all(exact == n).item() and torch.isfinite(half).all().item()):
        raise RuntimeError("Perhitungan uji di GPU salah atau non-finite; restart runtime atau ganti GPU.")
    del exact, x, half
    torch.cuda.empty_cache()
    return device


if PROFILE == "gpu":
    DEVICE = cuda_gate()
    props = torch.cuda.get_device_properties(DEVICE)
    free_b, total_b = torch.cuda.mem_get_info()
    # torch.cuda.is_bf16_supported() bisa True di T4 lewat emulasi; yang menentukan adalah dukungan native (cc >= 8).
    AMP_DTYPE = torch.bfloat16 if props.major >= 8 else torch.float16
    print(f"GPU {props.name} | compute capability {props.major}.{props.minor} | "
          f"memori bebas {free_b / 2**30:.1f} dari {total_b / 2**30:.1f} GiB")
else:
    DEVICE = torch.device("cpu")
    AMP_DTYPE = None
    print("PROFIL cpu_smoke (opt-in): semua di CPU dengan model mini acak; angka di bawah TIDAK bermakna.")

USE_SCALER = AMP_DTYPE == torch.float16
print(f"device {DEVICE} | autocast {AMP_DTYPE} | GradScaler {USE_SCALER}")


def amp_ctx():
    """Autocast untuk forward di GPU; di CPU tidak melakukan apa-apa (float32 penuh)."""
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def new_scaler():
    return torch.amp.GradScaler(DEVICE.type, enabled=USE_SCALER)


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

GPU Tesla T4 | compute capability 7.5 | memori bebas 14.4 dari 14.6 GiB
device cuda | autocast torch.float16 | GradScaler True


## 1. Fine-tuning dasar: GPT-2 di wikitext

Buku membuka bab dengan tiga fungsi: memuat model dan tokenizer, memuat dataset, lalu menjalankan `Trainer`. Datasetnya **wikitext-2**, kumpulan artikel Wikipedia. GPT-2 dilatih di teks web, jadi Wikipedia itu dekat tetapi bukan "rumahnya". Kasus seperti ini pas untuk fine-tuning: model sudah paham bahasa Inggris dan tinggal menyesuaikan diri dengan gaya teksnya.

> **Listing buku: memuat model dan dataset**
> ```python
> def load_model_and_tokenizer(model_name="gpt2"):
>     model = GPT2LMHeadModel.from_pretrained(model_name)
>     ...
> def prepare_dataset(dataset_name="wikitext", dataset_config="wikitext-2-raw-v1"):
>     dataset = load_dataset(dataset_name, dataset_config)
> ```
> Listing bab ini tidak menuliskan `import` untuk `GPT2LMHeadModel`, `load_dataset`, `Trainer`, dan kawan-kawannya, sehingga bila disalin apa adanya berhenti di `NameError`. Nama `"wikitext"` tanpa namespace juga ditolak `datasets` versi baru (`HfUriError: Repository id must be 'namespace/name'`, terverifikasi dengan `datasets` 5.1.0). Sel berikut memakai import lengkap dari sel setup, nama kanonik `Salesforce/wikitext` dengan revisi yang dikunci, dan memindahkan model ke device hasil gerbang.

Sel di bawah juga menyimpan salinan bobot awal (`SD0`) di RAM CPU. Setiap eksperimen dimulai dari titik yang sama lewat `reset_gpt2()`, yang mengembalikan bobot *dan* mencairkan semua parameter. Tanpa langkah kedua itu, status beku dari eksperimen freezing akan terbawa diam-diam ke eksperimen berikutnya.

In [3]:
MODEL_ID, MODEL_REV = "openai-community/gpt2", "607a30d783dfa663caf39e06633721c8d4cfcd7e"   # repo kanonik "gpt2"
WIKI_ID, WIKI_REV = "Salesforce/wikitext", "b08601e04326c79dfdd32d625aee71d232d685c3"


def load_model_and_tokenizer(model_name=MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_name, revision=MODEL_REV)
    tokenizer.pad_token = tokenizer.eos_token          # GPT-2 tidak punya token padding sendiri
    if PROFILE == "gpu":
        model = GPT2LMHeadModel.from_pretrained(model_name, revision=MODEL_REV)
    else:                                              # cpu_smoke: 12 blok seperti GPT-2, tetapi sempit dan acak
        model = GPT2LMHeadModel(GPT2Config(vocab_size=len(tokenizer), n_positions=256,
                                           n_embd=32, n_layer=12, n_head=2))
    return model.to(DEVICE, dtype=torch.float32), tokenizer   # bobot master float32 di device gerbang


def prepare_dataset(dataset_name=WIKI_ID, dataset_config="wikitext-2-raw-v1"):
    return load_dataset(dataset_name, dataset_config, revision=WIKI_REV)


model, tokenizer = load_model_and_tokenizer()
SD0 = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
N_TOTAL = sum(p.numel() for p in model.parameters())


def reset_gpt2():
    """Kembalikan bobot awal DAN cairkan semua parameter."""
    model.load_state_dict(SD0)
    for p in model.parameters():
        p.requires_grad_(True)
    model.zero_grad(set_to_none=True)


assert next(model.parameters()).device.type == DEVICE.type
assert next(model.parameters()).dtype == torch.float32
assert model.lm_head.weight is model.transformer.wte.weight      # output head berbagi bobot dengan embedding
print(f"model {MODEL_ID if PROFILE == 'gpu' else 'GPT-2 mini acak (cpu_smoke)'}: {N_TOTAL:,} parameter, "
      f"{len(model.transformer.h)} blok, di {next(model.parameters()).device}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model openai-community/gpt2: 124,439,808 parameter, 12 blok, di cuda:0


### Dari mana `labels` datang?

GPT-2 adalah model **kausal**: di setiap posisi ia menebak token berikutnya. Untuk training, Hugging Face meminta `labels` yang isinya sama dengan `input_ids`, lalu model sendiri yang menggeser satu posisi. Logit di posisi *t* dinilai terhadap token di posisi *t*+1.

Komplikasinya, panjang baris wikitext berbeda-beda, sedangkan satu batch harus berbentuk persegi. Buku memotong dan mengisi (*padding*) setiap baris sampai panjang tetap. GPT-2 tidak punya token padding, jadi token `<|endoftext|>` dipinjam sebagai pengisi. Posisi padding tidak boleh ikut dinilai, dan konvensinya adalah label **-100**, artinya "abaikan posisi ini".

> **Listing buku: tokenisasi dan `TrainingArguments`**
> ```python
> # di dalam tokenize_function
> return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
> training_args = TrainingArguments(output_dir="./fine_tuned_model", num_train_epochs=3,
>     per_device_train_batch_size=8, per_device_eval_batch_size=8,
>     warmup_steps=500, weight_decay=0.01, logging_dir="./logs")
> ```
> Dua masalah terverifikasi dengan `transformers` 5.20.2. Argumen `logging_dir` sudah tidak ada (`TypeError: ... unexpected keyword argument 'logging_dir'`). Lalu hasil tokenisasi hanya berisi `input_ids` dan `attention_mask`; tanpa *data collator* yang membuat `labels`, `trainer.train()` berhenti dengan `ValueError: The model did not return a loss from the inputs`. Sel berikut memakai `DataCollatorForLanguageModeling(mlm=False)`, yang menyalin `input_ids` menjadi `labels` dan mengganti posisi padding dengan -100.

Sel berikut juga membuang baris kosong. Dari 36.718 baris train wikitext-2, hanya 23.767 yang berisi teks. Baris kosong akan menjadi urutan yang seluruhnya padding, jadi seluruh labelnya -100 dan tidak ada yang bisa dipelajari darinya.

In [4]:
raw = prepare_dataset()
raw = raw.filter(lambda ex: len(ex["text"].strip()) > 0)   # baris kosong hanya akan berisi padding
small = datasets.DatasetDict(
    train=raw["train"].select(range(CFG["n_train_lines"])),
    validation=raw["validation"].select(range(CFG["n_val_lines"])),
)


def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True,
                     padding="max_length", max_length=CFG["seq_len"])   # buku: 512


tokenized = small.map(lambda ex: tokenize_function(ex, tokenizer), batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

lengths = [len(ids) for ids in tokenizer(list(small["train"]["text"]))["input_ids"]]
kept = sum(min(n, CFG["seq_len"]) for n in lengths)
print(f"baris non-kosong: train {len(raw['train']):,} | validation {len(raw['validation']):,}")
print(f"dipakai: train {len(tokenized['train'])} baris | validation {len(tokenized['validation'])} baris")
print(f"rata-rata {np.mean(lengths):.1f} token per baris | {np.mean([n > CFG['seq_len'] for n in lengths]):.1%} "
      f"baris terpotong di {CFG['seq_len']} token | token tersisa {kept:,} dari {sum(lengths):,} "
      f"({kept / sum(lengths):.0%})")

batch = collator([tokenized["train"][i] for i in range(2)])
mask = batch["attention_mask"].bool()
assert torch.equal(batch["labels"][mask], batch["input_ids"][mask])   # token asli: label = input
assert torch.all(batch["labels"][~mask] == -100)                      # padding: diabaikan
print(f"kolom batch dari collator: {list(batch.keys())}")
for row in range(2):
    print(f"baris #{row}: {int(mask[row].sum())} token asli, {int((~mask[row]).sum())} padding berlabel -100")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

baris non-kosong: train 23,767 | validation 2,461
dipakai: train 4000 baris | validation 400 baris
rata-rata 96.2 token per baris | 32.5% baris terpotong di 128 token | token tersisa 278,947 dari 384,932 (72%)
kolom batch dari collator: ['input_ids', 'attention_mask', 'labels']
baris #0: 9 token asli, 119 padding berlabel -100
baris #1: 128 token asli, 0 padding berlabel -100


Dua baris contoh memperlihatkan pembagian kerjanya. Baris pendek (judul artikel) terisi padding, dan setiap posisi padding diberi label -100. Baris panjang terisi penuh token asli, dan labelnya identik dengan input. Dua `assert` di atas memeriksa aturan itu untuk seluruh batch.

Angka pemotongan juga penting. Di profil `gpu`, 4.000 baris train rata-rata berisi 96 token, tetapi sekitar sepertiganya (32,5%) lebih panjang dari 128 token dan terpotong, sehingga hanya 72% token yang benar-benar dipakai. Batas 512 seperti di buku tidak memotong sebanyak itu, tetapi dengan rata-rata 96 token per baris, sekitar 80% isi setiap batch akan berupa padding yang tetap dihitung GPU. Bagian 4 memakai cara ketiga, *packing*, yang menghindari kedua pemborosan itu.

### Training dengan `Trainer`

Penggarisnya adalah **validation loss**: rata-rata cross-entropy per token pada baris validation yang tidak pernah dipakai untuk training, dalam satuan *nat*. Kita menghitungnya dengan fungsi kecil `eval_loss`, yang hanya menghitung token yang benar-benar dinilai (setelah pergeseran kausal dan tanpa padding). Fungsi yang sama dipakai lagi di bagian 2, jadi angkanya bisa dibandingkan langsung. Bentuk yang lebih mudah dibayangkan adalah **perplexity** = e^loss. Perplexity 40 kira-kira berarti model seragu orang yang harus memilih acak di antara 40 token. Makin kecil, makin baik model menebak teks itu.

Buku melatih 3 epoch penuh. Di sini cukup `CFG["trainer_steps"]` = 120 step × 8 baris = 960 baris, sekitar 4% dari baris berisi teks di wikitext-2, karena tujuannya melihat arah perubahan, bukan mengejar skor. `warmup_steps=500` dari buku lebih panjang daripada seluruh run, sehingga learning rate tidak akan pernah mencapai nilai targetnya; warmup di sini 10% dari jumlah step. Fungsi `make_trainer` meneruskan presisi dari gerbang (`fp16` di T4, `bf16` di GPU cc ≥ 8), tidak menyimpan checkpoint, tidak mengirim log ke layanan luar, dan berhenti bila `Trainer` memilih device selain hasil gerbang.

In [5]:
def make_trainer(name, model, train_dataset=None, eval_dataset=None, data_collator=None,
                 optimizers=(None, None), **overrides):
    """Trainer dengan device dan presisi yang sama dengan gerbang GPU; tanpa checkpoint dan logging eksternal."""
    args = dict(output_dir=os.path.join(WORK_DIR, name), report_to="none", save_strategy="no",
                disable_tqdm=True, logging_strategy="steps", logging_steps=10, seed=42,
                per_device_train_batch_size=CFG["batch_size"], per_device_eval_batch_size=CFG["batch_size"],
                fp16=AMP_DTYPE == torch.float16, bf16=AMP_DTYPE == torch.bfloat16,
                use_cpu=DEVICE.type == "cpu", dataloader_pin_memory=DEVICE.type == "cuda")
    args.update(overrides)
    targs = TrainingArguments(**args)
    if targs.device.type != DEVICE.type:
        raise RuntimeError(f"Trainer memilih {targs.device}, padahal gerbang memilih {DEVICE}")
    trainer = Trainer(model=model, args=targs, train_dataset=train_dataset, eval_dataset=eval_dataset,
                      data_collator=data_collator, optimizers=optimizers)
    trainer.remove_callback(PrinterCallback)        # ringkasan dicetak sendiri, bukan log mentah per step
    return trainer


def finish_trainer(trainer):
    """accelerate membungkus model.forward dengan autocast; lepas agar loop manual memakai amp_ctx() sendiri."""
    trainer.accelerator.unwrap_model(trainer.model, keep_fp32_wrapper=False)
    model.__dict__.pop("forward", None)
    model.__dict__.pop("_original_forward", None)
    free_memory()


def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


@torch.no_grad()
def eval_loss(model, batches):
    """Validation loss rata-rata per token yang dinilai (setelah geser kausal, tanpa padding)."""
    model.eval()
    total, count = 0.0, 0
    for batch in batches:
        batch = to_device(batch)
        with amp_ctx():
            loss = model(**batch).loss
        n = int((batch["labels"][:, 1:] != -100).sum())     # model menggeser label satu posisi
        total += float(loss) * n
        count += n
    return total / count


bs = CFG["batch_size"]
val_rows = tokenized["validation"]
val_batches = [collator([val_rows[i] for i in range(j, min(j + bs, len(val_rows)))])
               for j in range(0, len(val_rows), bs)]

reset_gpt2()
steps = CFG["trainer_steps"]
log_every = max(1, steps // 12)
trainer = make_trainer("wikitext_ft", model, train_dataset=tokenized["train"], data_collator=collator,
                       max_steps=steps, learning_rate=5e-5, weight_decay=0.01,
                       warmup_steps=max(1, steps // 10), logging_steps=log_every)
loss_before = eval_loss(model, val_batches)
train_out = trainer.train()
loss_after = eval_loss(model, val_batches)
assert next(trainer.model.parameters()).device.type == DEVICE.type
assert math.isfinite(loss_before) and math.isfinite(loss_after)

ratio = math.exp(loss_after - loss_before)
print(f"val loss sebelum : {loss_before:.3f}  (perplexity {math.exp(loss_before):.1f})")
print(f"val loss sesudah : {loss_after:.3f}  (perplexity {math.exp(loss_after):.1f})")
print(f"perubahan        : {loss_after - loss_before:+.3f} nat, perplexity x{ratio:.3f} ({ratio - 1:+.1%})")
print(f"training {steps} step x {CFG['batch_size']} baris = {steps * CFG['batch_size']} baris, "
      f"{float(train_out.metrics['train_runtime']):.1f} s di {trainer.args.device}")
train_losses = [float(h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"train loss tiap {log_every} step:", " ".join(f"{x:.2f}" for x in train_losses))
finish_trainer(trainer)
del trainer

val loss sebelum : 3.977  (perplexity 53.3)
val loss sesudah : 3.457  (perplexity 31.7)
perubahan        : -0.520 nat, perplexity x0.594 (-40.6%)
training 120 step x 8 baris = 960 baris, 15.5 s di cuda:0
train loss tiap 10 step: 4.28 4.10 3.95 3.82 3.80 3.72 3.66 3.62 3.74 3.66 3.63 3.73


Cara membaca tiga baris pertama: perubahan loss dalam nat langsung diterjemahkan ke perplexity, karena perplexity = e^loss. Penurunan loss sebesar Δ berarti perplexity dikalikan e^−Δ. Sebagai contoh hitung, Δ = 0,2 nat berarti perplexity menjadi 0,82 kali semula, turun sekitar 18%. Baris `perubahan` mencetak faktor itu untuk run Anda.

Bila loss turun, artinya model menjadi lebih yakin pada token wikitext yang *belum pernah dilihatnya*, setelah hanya membaca sekitar seribu baris. Itulah inti transfer learning: model yang sudah paham bahasa cukup disetel, tidak perlu diajari ulang. Klaimnya tetap sempit: perbaikan ini diukur pada 400 baris validation dari distribusi yang sama, bukan kualitas model secara umum. Train loss yang dicatat `Trainer` juga tidak bisa dibandingkan langsung dengan validation loss, karena dihitung pada data training dengan dropout aktif dan berfluktuasi antar-batch.

## 2. Membekukan dan mencairkan layer

GPT-2 124M terdiri dari embedding token dan posisi (39,4 juta parameter), 12 blok transformer (masing-masing 7,09 juta), layer norm akhir `ln_f`, dan output head `lm_head`. Heuristik buku: blok bawah menyimpan pola bahasa umum, blok atas lebih spesifik terhadap tugas. Jadi blok bawah dibekukan dan blok atas dilatih. **Membekukan** artinya `requires_grad=False`: autograd tidak menghitung gradien untuk parameter itu, dan optimizer tidak menyimpan state untuknya.

> **Listing buku: `freeze_layers`**
> ```python
> def freeze_layers(model, num_layers_to_freeze):
>     for param in model.base_model.parameters():
>         param.requires_grad = False
>     for i, layer in enumerate(model.base_model.transformer.h):
>         if i >= len(model.base_model.transformer.h) - num_layers_to_freeze:
>             for param in layer.parameters():
>                 param.requires_grad = True
> ```
> Pada `GPT2LMHeadModel`, `model.base_model` *sudah* berupa `GPT2Model`, sehingga atribut `.transformer` tidak ada: `AttributeError: 'GPT2Model' object has no attribute 'transformer'` (terverifikasi). Blok-bloknya ada di `model.transformer.h`. Nama argumennya juga menyesatkan: fungsi ini membekukan semuanya lalu *mencairkan* `num_layers_to_freeze` blok teratas, jadi `freeze_layers(model, 2)` justru melatih 2 blok dan membekukan 10. Versi di bawah memakai nama yang sesuai perilakunya, `freeze_all_but_top`.

Ada efek samping yang mudah terlewat. `lm_head` GPT-2 berbagi bobot (*tied*) dengan embedding token `wte`, jadi membekukan embedding berarti output head ikut beku. Sel berikut membuktikannya dan memeriksa gradien setelah satu backward: parameter beku tidak boleh punya gradien, dan semua parameter yang dilatih harus punya gradien finite.

In [6]:
def freeze_all_but_top(model, n_top):
    """Bekukan semua parameter, lalu latih hanya n_top blok teratas plus layer norm akhir (ln_f)."""
    for p in model.parameters():
        p.requires_grad_(False)
    blocks = model.transformer.h
    for block in blocks[len(blocks) - n_top:]:
        for p in block.parameters():
            p.requires_grad_(True)
    for p in model.transformer.ln_f.parameters():
        p.requires_grad_(True)


def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def fixed_batches(n_batches, offset=0):
    """Batch deterministik dari baris train wikitext, supaya setiap strategi melihat data yang sama."""
    bs = CFG["batch_size"]
    assert offset + n_batches * bs <= len(tokenized["train"]), "baris train kurang untuk jumlah step ini"
    return [collator([tokenized["train"][i] for i in range(offset + j * bs, offset + (j + 1) * bs)])
            for j in range(n_batches)]


reset_gpt2()
freeze_all_but_top(model, 2)
model.train()
with amp_ctx():
    loss = model(**to_device(fixed_batches(1)[0])).loss
loss.backward()
frozen = [p for p in model.parameters() if not p.requires_grad]
trained = [p for p in model.parameters() if p.requires_grad]
assert all(p.grad is None for p in frozen), "parameter beku tidak boleh punya gradien"
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in trained)
assert sum(p.grad.abs().sum().item() for p in trained) > 0
print(f"dilatih {n_trainable(model):,} dari {N_TOTAL:,} parameter ({n_trainable(model) / N_TOTAL:.1%})")
print(f"tensor parameter: {len(trained)} dilatih (semua bergradien finite), {len(frozen)} beku (tanpa gradien)")
print(f"lm_head ikut beku karena berbagi bobot dengan wte: {not model.lm_head.weight.requires_grad}")
reset_gpt2()

dilatih 14,177,280 dari 124,439,808 parameter (11.4%)
tensor parameter: 26 dilatih (semua bergradien finite), 122 beku (tanpa gradien)
lm_head ikut beku karena berbagi bobot dengan wte: True


Di GPT-2 124M, dua blok teratas plus `ln_f` berarti 14.177.280 parameter dilatih, 11,4% dari 124.439.808. Angka ini pasti, karena hanya bergantung pada arsitektur. Pertanyaan berikutnya: apakah 89% parameter yang tidak dilatih itu benar-benar menghemat sesuatu?

### Eksperimen: freezing diukur dari empat sisi

Tiga strategi dibandingkan: full fine-tune, top-6 + `ln_f`, dan top-2 + `ln_f`. Ketiganya mulai dari bobot pretrained yang sama, melihat batch yang sama dalam urutan sama, dengan masker dropout yang sama, dan memakai AdamW yang hanya diberi parameter yang dilatih. Yang diukur adalah jumlah parameter dilatih, milidetik per step, memori GPU puncak, dan penurunan validation loss.

Perkiraan memorinya bisa dihitung lebih dulu. Setiap parameter yang dilatih membawa gradien dan dua momen AdamW, masing-masing float32, jadi 12 byte. Full fine-tune membutuhkan 124,4 juta × 12 byte ≈ 1,5 GB di luar bobot model (0,5 GB), sedangkan top-2 hanya 14,2 juta × 12 byte ≈ 0,17 GB. Memori puncak yang diukur juga memuat aktivasi, jadi selisihnya tidak akan persis sama dengan perkiraan ini.

Waktu per step diukur dengan `synchronize` sebelum dan sesudah setiap step, karena GPU bekerja asinkron. Step pertama tidak dihitung karena ikut menanggung biaya inisialisasi, dan yang dilaporkan adalah median.

In [7]:
def train_on(model, optimizer, batches, scaler):
    """Loop training manual: autocast + GradScaler, waktu per step diukur dengan synchronize."""
    model.train()
    times, skipped = [], 0
    for batch in batches:
        batch = to_device(batch)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with amp_ctx():
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise FloatingPointError(f"loss non-finite: {loss.item()}")
        scaler.scale(loss).backward()
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        skipped += int(USE_SCALER and scaler.get_scale() < scale_before)   # step dilewati karena overflow fp16
    return {"ms": 1000 * float(np.median(times[1:] or times)), "skipped": skipped}


base_val = loss_before                                  # bobot awal yang sama dengan sebelum training bagian 1
batches = fixed_batches(CFG["arm_steps"])
rows = []
for name, n_top in (("full fine-tune", None), ("top-6 + ln_f", 6), ("top-2 + ln_f", 2)):
    reset_gpt2()
    if n_top is not None:
        freeze_all_but_top(model, n_top)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.01)
    free_memory()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    torch.manual_seed(0)                                   # masker dropout sama di semua strategi
    stats = train_on(model, optimizer, batches, new_scaler())
    peak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")
    val = eval_loss(model, val_batches)
    rows.append((name, n_trainable(model), stats["ms"], peak, val, stats["skipped"]))
    del optimizer
reset_gpt2()
free_memory()

steps = CFG["arm_steps"]
print(f"val loss awal (bobot SD0) {base_val:.3f} | {steps} step per strategi, "
      f"batch {CFG['batch_size']} x {CFG['seq_len']} token")
print(f"{'strategi':<15}{'dilatih':>10}{'ms/step':>9}{'memori puncak':>15}{'val loss':>10}"
      f"{'turun':>8}{'turun/detik':>13}{'step dilewati':>15}")
for name, tr, ms, peak, val, skipped in rows:
    drop = base_val - val
    print(f"{name:<15}{tr / 1e6:>9.1f}M{ms:>9.0f}{peak:>11.0f} MiB{val:>10.3f}{drop:>8.3f}"
          f"{drop / (steps * ms / 1000):>13.3f}{skipped:>15}")

val loss awal (bobot SD0) 3.977 | 30 step per strategi, batch 8 x 128 token
strategi          dilatih  ms/step  memori puncak  val loss   turun  turun/detik  step dilewati
full fine-tune     124.4M      145       3101 MiB     3.735   0.242        0.055              4
top-6 + ln_f        42.5M       86       1977 MiB     3.709   0.267        0.104              0
top-2 + ln_f        14.2M       61       1431 MiB     3.794   0.183        0.100              0


Cara membaca tabelnya, kolom demi kolom:

- **dilatih** pasti: 124,4M, 42,5M, dan 14,2M. Top-2 melatih 11,4% parameter, top-6 34,2%.
- **memori puncak** sebaiknya dibandingkan dengan perkiraan 12 byte per parameter dilatih di atas. Selisih antara full dan top-2 seharusnya mendekati 1,3 GB; sisanya adalah bobot model dan aktivasi yang sama di ketiga strategi.
- **ms/step** turun karena backward berhenti di blok terbawah yang masih dilatih, dan gradien bobot `lm_head` yang besar (768 × 50.257) tidak dihitung. Forward tetap penuh. Dengan batch sekecil 8 × 128 token, sebagian waktu step adalah biaya tetap (peluncuran kernel, langkah optimizer), jadi penghematannya bisa lebih kecil dari rasio blok.
- **turun** adalah penurunan validation loss setelah jumlah step yang sama. Secara teori full fine-tune turun paling dalam per step, karena semua parameter ikut menyesuaikan diri. **turun/detik** membagi penurunan itu dengan waktu compute, dan di kolom inilah strategi beku bisa bersaing.
- **step dilewati** hanya relevan untuk float16. Bila GradScaler mendeteksi overflow, step itu dibatalkan dan skalanya diturunkan. Angka di atas nol berarti strategi itu mendapat lebih sedikit update efektif.

Kesimpulan yang aman bergantung pada angka Anda. Bila top-2 menghemat memori sesuai perkiraan dan turun/detik-nya mendekati full fine-tune, freezing menjadi pilihan masuk akal saat memori atau waktu terbatas. Bila penurunan loss-nya tertinggal jauh, harga penghematan itu adalah adaptasi yang lebih dangkal.

### Gradual unfreezing: optimizer harus ikut tahu

Buku juga memberi `gradual_unfreeze`: mencairkan lebih banyak layer setiap epoch sambil memanggil `trainer.train(resume_from_checkpoint=True)` berulang kali. Ada dua jebakan di pola itu. Pertama, `resume_from_checkpoint=True` menuntut checkpoint sudah ada di `output_dir`. Kedua, dan lebih berbahaya karena tidak ada error, `Trainer` membuat optimizer pada pemanggilan `train()` pertama lalu memakainya lagi. Parameter yang dicairkan sesudahnya tidak pernah masuk ke optimizer itu. Dalam uji lokal dengan GPT-2 mini, setelah semua layer dicairkan dan `train()` dipanggil lagi, optimizer tetap memegang parameter tahap pertama saja dan bobot blok terbawah tidak berubah sama sekali.

Perbaikannya adalah loop sendiri. Setiap kali ada layer yang dicairkan, parameternya dimasukkan ke optimizer dengan `add_param_group`. Momen Adam milik parameter yang sudah dilatih tetap utuh, dan `assert` memastikan isi optimizer selalu sama persis dengan himpunan parameter yang dilatih.

In [8]:
def sync_optimizer(optimizer, model):
    """Masukkan parameter yang baru dicairkan ke optimizer; state Adam parameter lama tetap utuh."""
    known = {id(p) for group in optimizer.param_groups for p in group["params"]}
    new = [p for p in model.parameters() if p.requires_grad and id(p) not in known]
    if new:
        optimizer.add_param_group({"params": new})
    return sum(p.numel() for p in new)


reset_gpt2()
stages = (2, 6, 12)
freeze_all_but_top(model, stages[0])
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.01)
scaler = new_scaler()
torch.manual_seed(0)
print(f"val loss awal (bobot SD0) {base_val:.3f}")
print(f"{'tahap':<7}{'blok cair':>10}{'baru masuk':>12}{'di optimizer':>14}{'dilatih':>10}{'val loss':>10}")
for k, n_top in enumerate(stages):
    freeze_all_but_top(model, n_top)
    added = sync_optimizer(optimizer, model)
    in_opt = {id(p) for group in optimizer.param_groups for p in group["params"]}
    assert in_opt == {id(p) for p in model.parameters() if p.requires_grad}, "optimizer tidak sinkron"
    n_opt = sum(p.numel() for group in optimizer.param_groups for p in group["params"])
    offset = k * CFG["stage_steps"] * CFG["batch_size"]
    train_on(model, optimizer, fixed_batches(CFG["stage_steps"], offset=offset), scaler)
    print(f"{k + 1:<7}{n_top:>10}{(added if k else n_opt) / 1e6:>11.1f}M{n_opt / 1e6:>13.1f}M"
          f"{n_trainable(model) / 1e6:>9.1f}M{eval_loss(model, val_batches):>10.3f}")
del optimizer
reset_gpt2()
free_memory()

val loss awal (bobot SD0) 3.977
tahap   blok cair  baru masuk  di optimizer   dilatih  val loss
1               2       14.2M         14.2M     14.2M     3.834
2               6       28.4M         42.5M     42.5M     3.700
3              12       42.5M         85.1M     85.1M     3.616


Kolom **di optimizer** harus selalu sama dengan **dilatih**; kalau tidak, `assert` menghentikan sel. Di tahap ketiga, semua 12 blok sudah cair (85,1 juta parameter), tetapi embedding 39,4 juta tetap beku karena `freeze_all_but_top` tidak pernah mencairkannya. Kolom **val loss** memperlihatkan efek kumulatif tiga tahap, masing-masing `CFG["stage_steps"]` step pada batch yang berbeda. Angka ini tidak sebanding langsung dengan tabel sebelumnya, karena jumlah step dan batch-nya lain.

## 3. Learning-rate schedule

Pada step-step awal, estimasi momen Adam masih kasar, sehingga langkah besar bisa merusak bobot pretrained yang sudah bagus. **Warmup** menaikkan learning rate secara linear dari 0 ke nilai target selama beberapa step pertama. Sesudahnya learning rate diturunkan sampai 0, entah secara **linear** atau mengikuti kurva **cosine**, supaya langkah-langkah terakhir kecil dan halus.

> **Listing buku: menghitung step dan membuat scheduler**
> ```python
> num_training_steps = len(tokenized_dataset["train"]) //
> training_args.per_device_train_batch_size * num_epochs
> scheduler = get_linear_schedule_with_warmup(
>     trainer.optimizer, num_warmup_steps=num_training_steps // 10,
>     num_training_steps=num_training_steps)
> ```
> Baris pertama patah setelah `//` tanpa tanda kurung, jadi `SyntaxError`. Setelah disambung, baris berikutnya gagal karena `Trainer` baru membuat optimizer di dalam `train()`. Sebelum itu `trainer.optimizer` masih `None` (`AttributeError: 'NoneType' object has no attribute 'param_groups'`, terverifikasi). Scheduler buatan sendiri juga baru dipakai bila diserahkan ke `Trainer` lewat `optimizers=(optimizer, scheduler)`. Cara lainnya adalah membiarkan `Trainer` membuatnya sendiri lewat `lr_scheduler_type`. Kedua cara dipakai di bawah.

Pertama, bentuk kurvanya. Sel ini murni aritmetika jadwal di CPU dengan satu parameter boneka, tanpa model dan tanpa GPU.

In [9]:
def lr_trace(make_schedule, total=100, warmup=10, base_lr=2e-5):
    dummy = torch.optim.SGD([torch.nn.Parameter(torch.zeros(1))], lr=base_lr)
    schedule = make_schedule(dummy, warmup, total)
    lrs = []
    for _ in range(total + 1):
        lrs.append(dummy.param_groups[0]["lr"])     # LR yang dipakai pada step ini
        dummy.step()                                # urutan benar: optimizer dulu, baru scheduler
        schedule.step()
    return lrs


curves = {"linear": lr_trace(get_linear_schedule_with_warmup),
          "cosine": lr_trace(get_cosine_schedule_with_warmup)}
probe = (0, 5, 10, 25, 50, 75, 100)
print(f"{'step':>7}" + "".join(f"{s:>10}" for s in probe))
for name, lrs in curves.items():
    print(f"{name:>7}" + "".join(f"{lrs[s]:>10.2e}" for s in probe))

   step         0         5        10        25        50        75       100
 linear  0.00e+00  1.00e-05  2.00e-05  1.67e-05  1.11e-05  5.56e-06  0.00e+00
 cosine  0.00e+00  1.00e-05  2.00e-05  1.87e-05  1.17e-05  3.57e-06  0.00e+00


Angka di tabel ini deterministik, jadi sama di mesin mana pun. Kedua jadwal naik identik selama warmup dan mencapai 2e-5 di step 10. Setelah itu bentuknya berpisah. Di step 25, cosine masih 1,87e-5 sementara linear sudah 1,67e-5, sekitar 12% lebih tinggi. Di step 50 keduanya hampir sama (1,17e-5 dan 1,11e-5). Di step 75 urutannya terbalik: cosine 3,57e-6, linear 5,56e-6, sehingga cosine kini 36% lebih rendah. Artinya, cosine mempertahankan langkah besar lebih lama lalu mendingin lebih tegas di akhir. Untuk fine-tuning pendek perbedaan ini kecil. Yang jauh lebih penting adalah jadwalnya benar-benar dipakai.

Sel berikut menyerahkan optimizer dan scheduler buatan sendiri ke `Trainer`, lalu memeriksa tiga hal: objek yang dipakai `Trainer` memang objek kita, scheduler benar-benar melangkah, dan learning rate yang tercatat mengikuti kurva.

In [10]:
reset_gpt2()
n_steps = CFG["sched_steps"]
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, n_steps // 10),
                                            num_training_steps=n_steps)
trainer = make_trainer("schedule_demo", model, train_dataset=tokenized["train"], data_collator=collator,
                       optimizers=(optimizer, scheduler), max_steps=n_steps, logging_steps=1)
trainer.train()
same_objects = (getattr(trainer.optimizer, "optimizer", trainer.optimizer) is optimizer
                and getattr(trainer.lr_scheduler, "scheduler", trainer.lr_scheduler) is scheduler)
logged = [float(h["learning_rate"]) for h in trainer.state.log_history if "learning_rate" in h]
assert same_objects, "Trainer memakai optimizer/scheduler lain"
assert 0 < scheduler.last_epoch <= n_steps and max(logged) <= 5e-5 * (1 + 1e-6)
print(f"optimizer dan scheduler di dalam Trainer adalah objek kita: {same_objects}")
print(f"scheduler melangkah {scheduler.last_epoch} kali dalam {n_steps} step training")
print("LR tercatat per step:", " ".join(f"{x:.1e}" for x in logged))
finish_trainer(trainer)
del trainer, optimizer, scheduler

optimizer dan scheduler di dalam Trainer adalah objek kita: True
scheduler melangkah 20 kali dalam 20 step training
LR tercatat per step: 0.0e+00 2.5e-05 5.0e-05 5.0e-05 4.8e-05 4.7e-05 4.4e-05 4.1e-05 3.8e-05 3.4e-05 2.9e-05 2.5e-05 2.1e-05 1.6e-05 1.3e-05 8.9e-06 5.8e-06 3.3e-06 1.5e-06 3.8e-07


Deret LR yang tercatat seharusnya naik selama warmup (2 step di profil `gpu`), mencapai 5e-5, lalu turun mengikuti cosine. `Trainer` membungkus optimizer kita dengan lapisan dari `accelerate`, tetapi objek di dalamnya tetap objek kita, sehingga jadwal yang kita rancang memang yang dipakai. Di GPU float16 ada satu catatan: bila GradScaler membatalkan sebuah step karena overflow, scheduler juga tidak melangkah pada step itu, sehingga jumlah langkah scheduler bisa sedikit di bawah jumlah step training.

## 4. Adaptasi domain: dari Wikipedia ke teks sains

Adaptasi domain berarti melanjutkan training bahasa pada teks mentah dari domain sasaran. Tidak perlu label manual, karena token berikutnya sudah menjadi labelnya. Model jadi terbiasa dengan kosakata dan gaya domain itu. Buku memakai korpus saintifik dari sebuah file teks. Di sini kita memakai paragraf penjelas (`support`) dari dataset **SciQ**: soal sains sekolah tentang biologi, kimia, fisika, dan ilmu bumi, masing-masing dengan paragraf penjelasnya. Lisensinya CC BY-NC 3.0, jadi boleh untuk belajar tetapi tidak untuk produk komersial.

> **Listing buku: dataset domain dan argumen training**
> ```python
> from transformers import TextDataset, DataCollatorForLanguageModeling
> dataset = TextDataset(tokenizer=tokenizer, file_path=file_path, block_size=128)
> training_args = TrainingArguments(output_dir="./scientific_model", ..., warmup_steps=1000,
>     weight_decay=0.1, learning_rate=3e-5, evaluation_strategy="steps",
>     eval_steps=500, save_steps=500, gradient_accumulation_steps=4)
> ```
> `TextDataset` sudah dihapus dari `transformers` 5 (`ImportError`), dan `evaluation_strategy` sudah berganti nama menjadi `eval_strategy` (`TypeError` untuk nama lama). Keduanya terverifikasi dengan 5.20.2. `DataCollatorForLanguageModeling` masih ada dan tetap dipakai. Pengganti `TextDataset` di bawah memakai `datasets`: baca file, tokenisasi, sambung, lalu potong per blok 128 token.

Sel pertama hanya pekerjaan I/O di CPU: mengunduh SciQ dan menulis paragrafnya ke file teks, satu paragraf per baris, supaya alurnya tetap "dari file" seperti di buku. Paragraf train yang juga muncul di validation dibuang lebih dulu, agar tidak ada teks evaluasi yang ikut dilatih.

In [11]:
# Bagian I/O di CPU: unduh SciQ, tulis paragraf penjelasnya ke file teks (satu paragraf per baris).
SCIQ_ID, SCIQ_REV = "allenai/sciq", "2c94ad3e1aafab77146f384e23536f97a4849815"   # lisensi CC BY-NC 3.0
sciq = load_dataset(SCIQ_ID, revision=SCIQ_REV)


def clean_paragraphs(split):
    return [s.strip().replace("\n", " ") for s in sciq[split]["support"] if s.strip()]


val_paras = clean_paragraphs("validation")
train_all = clean_paragraphs("train")
val_set = set(val_paras)
train_paras = [p for p in train_all if p not in val_set]     # cegah teks validation bocor ke training
n_removed = len(train_all) - len(train_paras)
if PROFILE == "cpu_smoke":
    train_paras, val_paras = train_paras[:64], val_paras[:16]

paths = {}
for split, paras in (("train", train_paras), ("validation", val_paras)):
    paths[split] = os.path.join(WORK_DIR, f"sciq_support_{split}.txt")
    with open(paths[split], "w", encoding="utf-8") as f:
        f.write("\n".join(paras) + "\n")
print(f"paragraf SciQ: train {len(train_paras):,} (dibuang {n_removed} yang juga ada di validation) | "
      f"validation {len(val_paras):,}")
print("contoh:", train_paras[0][:200], "...")

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

paragraf SciQ: train 10,480 (dibuang 1 yang juga ada di validation) | validation 887
contoh: Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth tem ...


**Packing** adalah pengganti `TextDataset`. Semua paragraf disambung menjadi satu aliran token panjang, lalu dipotong per blok 128 token. Bandingkan dengan bagian 1: tidak ada padding dan tidak ada token yang terbuang karena pemotongan, sehingga setiap posisi di setiap blok ikut dinilai. Harganya, satu paragraf bisa terbelah di dua blok, dan untuk training bahasa itu tidak masalah. Untuk SciQ di profil `gpu`, 10.480 paragraf train menjadi sekitar 1,03 juta token, yang terpotong menjadi 8.074 blok; validation menjadi 675 blok, dan 200 di antaranya dipakai untuk evaluasi.

Sel ini juga memperlihatkan bagaimana tokenizer GPT-2 menangani istilah sains. Tokenizer byte-level BPE tidak pernah menghasilkan token "tak dikenal"; istilah yang jarang ia lihat hanya dipecah menjadi beberapa potongan.

In [12]:
def prepare_domain_dataset(file_path, tokenizer, block_size=128):
    """Pengganti TextDataset: baca file -> tokenisasi -> sambung jadi satu aliran token -> potong per blok."""
    raw_text = load_dataset("text", data_files=file_path, split="train")
    tok = raw_text.map(lambda ex: tokenizer([t + "\n\n" for t in ex["text"]]),
                       batched=True, remove_columns=["text"])

    def group(ex):
        ids = [t for seq in ex["input_ids"] for t in seq]
        n = len(ids) // block_size * block_size
        blocks = [ids[i:i + block_size] for i in range(0, n, block_size)]
        return {"input_ids": blocks, "attention_mask": [[1] * block_size for _ in blocks]}

    dataset = tok.map(group, batched=True, remove_columns=tok.column_names)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    return dataset, data_collator


domain_train, domain_collator = prepare_domain_dataset(paths["train"], tokenizer, CFG["seq_len"])
domain_val, _ = prepare_domain_dataset(paths["validation"], tokenizer, CFG["seq_len"])
domain_val = domain_val.select(range(min(len(domain_val), CFG["domain_val_blocks"])))
assert all(len(x) == CFG["seq_len"] for x in domain_train[:4]["input_ids"])
print(f"blok {CFG['seq_len']} token: train {len(domain_train):,} | validation dipakai {len(domain_val)}")
for term in (" photosynthesis", " mitochondria", " electronegativity"):
    print(f"{term.strip()!r:>20} -> {tokenizer.tokenize(term)}")

blok 128 token: train 8,074 | validation dipakai 200
    'photosynthesis' -> ['Ġphotos', 'ynthesis']
      'mitochondria' -> ['Ġmitochond', 'ria']
 'electronegativity' -> ['Ġelectr', 'one', 'g', 'ativity']


Hasil tokenizer di atas pasti: *photosynthesis* dan *mitochondria* masing-masing menjadi 2 token, *electronegativity* menjadi 4. Adaptasi domain membuat model terbiasa merangkai potongan-potongan seperti itu.

Argumen training mengikuti buku: learning rate 3e-5, weight decay 0,1, dan `gradient_accumulation_steps=4`. Gradient accumulation menjumlahkan gradien dari 4 batch sebelum optimizer melangkah, sehingga satu update melihat 8 × 4 = 32 blok (4.096 token) walau memori GPU hanya menampung 8 blok sekaligus. Dengan 60 step optimizer, model melihat 1.920 blok, sekitar 24% dari data train SciQ.

Tiga argumen buku diubah karena ukuran run. `warmup_steps=1000` diganti warmup 10% dengan kurva cosine, karena 1.000 step warmup lebih panjang dari seluruh run. `eval_steps=500` diganti evaluasi di tengah dan di akhir run, dengan nama argumen yang benar, `eval_strategy`. `save_steps` diganti `save_strategy="no"`, jadi tidak ada checkpoint yang ditulis.

Evaluasinya dua arah. **SciQ validation** mengukur adaptasi ke domain. **Wikitext validation**, data yang sama dengan bagian 1, mengukur apa yang terjadi pada kemampuan umum.

In [13]:
reset_gpt2()
steps = CFG["domain_steps"]
trainer = make_trainer("scientific_model", model, train_dataset=domain_train,
                       eval_dataset={"sciq": domain_val, "wiki": tokenized["validation"]},
                       data_collator=domain_collator, max_steps=steps,
                       learning_rate=3e-5, weight_decay=0.1, gradient_accumulation_steps=4,
                       warmup_steps=max(1, steps // 10), lr_scheduler_type="cosine",
                       eval_strategy="steps", eval_steps=max(1, steps // 2), logging_steps=max(1, steps // 2))
before = trainer.evaluate()                          # titik awal; train() memulai log baru, jadi disimpan sendiri
trainer.train()
by_step = collections.defaultdict(dict)                # tiap eval set tercatat sebagai entri log terpisah
by_step[0] = {key: float(before[key]) for key in ("eval_sciq_loss", "eval_wiki_loss")}
for h in trainer.state.log_history:
    for key in ("eval_sciq_loss", "eval_wiki_loss"):
        if key in h:
            by_step[h["step"]][key] = float(h[key])
history = [(step, v["eval_sciq_loss"], v["eval_wiki_loss"]) for step, v in sorted(by_step.items())]
assert len(history) >= 2 and all(math.isfinite(x) for _, a, b in history for x in (a, b))

print(f"{'step':>5}{'SciQ val loss':>15}{'wikitext val loss':>19}")
for step, sciq_loss, wiki_loss in history:
    print(f"{step:>5}{sciq_loss:>15.3f}{wiki_loss:>19.3f}")
d_sciq = history[-1][1] - history[0][1]
d_wiki = history[-1][2] - history[0][2]
seen = steps * CFG["batch_size"] * 4
print(f"perubahan: SciQ {d_sciq:+.3f} nat (perplexity x{math.exp(d_sciq):.3f}) | "
      f"wikitext {d_wiki:+.3f} nat (perplexity x{math.exp(d_wiki):.3f})")
print(f"{seen:,} blok dilihat ({seen / len(domain_train):.0%} dari train SciQ), batch efektif "
      f"{CFG['batch_size'] * 4} blok per step optimizer")
finish_trainer(trainer)
del trainer

 step  SciQ val loss  wikitext val loss
    0          3.421              4.014
   30          3.158              4.092
   60          3.133              4.098
perubahan: SciQ -0.288 nat (perplexity x0.750) | wikitext +0.083 nat (perplexity x1.087)
1,920 blok dilihat (24% dari train SciQ), batch efektif 32 blok per step optimizer


Baca kolom SciQ dulu. Penurunan loss di sana adalah adaptasi domain: model makin terbiasa dengan kalimat seperti "the cell membrane regulates…", padahal semua angka itu diukur pada paragraf yang tidak pernah dilatih. Lalu baca kolom wikitext, yaitu efek sampingnya. Bila loss wikitext naik sementara SciQ turun, model menukar sebagian kecocokan umum dengan kecocokan domain. Itu versi ringan dari *catastrophic forgetting* yang diukur langsung di bagian 6. Bila wikitext hampir tidak bergerak, pada skala run ini adaptasi domain tidak merusak kemampuan umum.

Angka di tabel ini dihitung oleh `Trainer` sendiri, yang merata-rata per batch dan, tergantung versinya, bisa ikut menghitung posisi pertama setiap baris sebagai token yang dinilai. Karena itu angka wikitext di step 0 bisa sedikit berbeda dari `eval_loss` di bagian 1 untuk bobot yang sama. Bandingkan angka di dalam tabel ini saja.

Saran domain buku yang lain tetap relevan, tetapi tidak diukur di sini: metrik evaluasi khusus domain dan penyesuaian kosakata untuk istilah teknis. Untuk GPT-2, "penyesuaian kosakata" bukan soal token tak dikenal, karena tidak ada token seperti itu. Masalahnya adalah istilah yang dipecah menjadi banyak potongan, sehingga memakan panjang konteks.

## 5. Few-shot dan zero-shot

Tugas konkretnya: diberi nama negara, jawab ibu kotanya. Ada tiga cara mendapatkan perilaku ini dari GPT-2, dan ketiganya diukur pada 16 negara *held-out* yang tidak muncul di contoh mana pun:

- **zero-shot**: langsung bertanya dengan format `Input: France\nOutput:` tanpa contoh;
- **few-shot in-context**: lima contoh terjawab ditaruh di prompt sebelum pertanyaan, tanpa mengubah bobot sama sekali;
- **few-shot fine-tuning** (SFT): bobot dilatih pada 16 pasangan contoh, lalu model ditanya tanpa contoh di prompt.

Jawaban dianggap benar bila kata pertama yang dihasilkan sama persis dengan nama ibu kota. Satu negara bernilai 1/16 = 6,25 poin persentase, jadi selisih satu atau dua negara masih dalam batas kebetulan.

> **Listing buku: `tokenize_few_shot`**
> ```python
> tokenized_prompt['labels'] = \
>     [-100] * len(tokenized_prompt['input_ids'])
> + tokenized_output['input_ids']
> ```
> Backslash hanya menyambung baris pertama, sehingga baris `+ ...` berdiri sendiri sebagai *unary plus* atas list: `TypeError: bad operand type for unary +: 'list'`. Seandainya baris itu diam-diam diabaikan, semua label bernilai -100, dan loss menjadi `nan` karena tidak ada satu token pun yang dinilai (terverifikasi). Bila `+`-nya disambung, desainnya tetap salah. Prompt dan jawaban ditokenisasi terpisah dan masing-masing di-pad ke 64, sehingga `labels` 128 token sedangkan input hanya 64 (`ValueError: Expected input batch_size (64) to match target batch_size (128)`). Pada model kausal, jawaban harus ada *di dalam* `input_ids` (prompt + jawaban), dan `labels` sepanjang itu juga, dengan -100 menutupi bagian prompt. Format itu dipakai di bawah.

In [14]:
TRAIN_PAIRS = [("Germany", "Berlin"), ("Spain", "Madrid"), ("Portugal", "Lisbon"), ("Netherlands", "Amsterdam"),
               ("Austria", "Vienna"), ("Poland", "Warsaw"), ("Sweden", "Stockholm"), ("Norway", "Oslo"),
               ("Finland", "Helsinki"), ("Greece", "Athens"), ("Turkey", "Ankara"), ("Thailand", "Bangkok"),
               ("Vietnam", "Hanoi"), ("Peru", "Lima"), ("Chile", "Santiago"), ("Kenya", "Nairobi")]
HELD_OUT = [("France", "Paris"), ("Japan", "Tokyo"), ("Italy", "Rome"), ("Egypt", "Cairo"),
            ("Canada", "Ottawa"), ("Russia", "Moscow"), ("China", "Beijing"), ("Ireland", "Dublin"),
            ("Indonesia", "Jakarta"), ("Belgium", "Brussels"), ("Denmark", "Copenhagen"), ("Hungary", "Budapest"),
            ("Cuba", "Havana"), ("Iran", "Tehran"), ("Iraq", "Baghdad"), ("Australia", "Canberra")]
assert not {c for c, _ in TRAIN_PAIRS} & {c for c, _ in HELD_OUT}, "negara held-out tidak boleh ada di contoh"


def query(country):
    return f"Input: {country}\nOutput:"


SHOTS = "".join(f"{query(c)} {k}\n\n" for c, k in TRAIN_PAIRS[:5])     # 5 contoh in-context


@torch.no_grad()
def ask(prompt_text, max_new_tokens=4):
    model.eval()
    enc = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    with amp_ctx():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def first_word(text):
    words = text.split("\n")[0].split()
    return words[0].strip(".,;:!?\"'") if words else ""


def accuracy(pairs, context=""):
    answers = [ask(context + query(c)) for c, _ in pairs]
    hits = sum(first_word(a) == k for a, (_, k) in zip(answers, pairs))
    return hits, answers


reset_gpt2()
n = len(HELD_OUT)
zero_hits, zero_ans = accuracy(HELD_OUT)
icl_hits, icl_ans = accuracy(HELD_OUT, context=SHOTS)
print(f"panjang prompt: zero-shot {len(tokenizer(query('France'))['input_ids'])} token | "
      f"5-shot {len(tokenizer(SHOTS + query('France'))['input_ids'])} token")
print(f"zero-shot         : {zero_hits}/{n} ({zero_hits / n:.0%})")
print(f"5-shot in-context : {icl_hits}/{n} ({icl_hits / n:.0%})")
for (c, _), z, i in list(zip(HELD_OUT, zero_ans, icl_ans))[:5]:
    print(f"  {c:<10} zero-shot {z!r:<24} 5-shot {i!r}")

panjang prompt: zero-shot 6 token | 5-shot 51 token
zero-shot         : 0/16 (0%)
5-shot in-context : 7/16 (44%)
  France     zero-shot ' France\n\nOutput'      5-shot ' Paris\n\nInput'
  Japan      zero-shot ' Japan\n\nOutput'       5-shot ' Tokyo\n\nInput'
  Italy      zero-shot ' Italy\n\nOutput'       5-shot ' Milan\n\nInput'
  Egypt      zero-shot ' Egypt\n\nOutput'       5-shot ' Cairo\n\nInput'
  Canada     zero-shot ' Canada\n\nOutput'      5-shot ' Toronto\n\nInput'


Baris jawaban mentah sama pentingnya dengan skornya. Pada zero-shot, kegagalan GPT-2 biasanya soal format: ia tidak tahu bahwa setelah `Output:` yang diharapkan adalah satu nama kota, sehingga sering mengulang nama negara atau berpindah baris. Lima contoh di prompt memperlihatkan formatnya. Bila skor 5-shot jauh di atas zero-shot, pengetahuan ibu kota sebenarnya sudah ada di bobot pretrained dan hanya perlu "dipanggil" dengan format yang benar. Harganya terlihat di baris panjang prompt: setiap pertanyaan 5-shot membawa puluhan token konteks tambahan, untuk setiap pemanggilan.

SFT memindahkan format itu ke dalam bobot. Sel berikut membangun contoh training dengan format kausal yang benar. Token `<|endoftext|>` ditambahkan di akhir jawaban *dan ikut dinilai*, supaya model belajar kapan berhenti. Label dibuat manual, bukan lewat collator: karena padding memakai token yang sama dengan EOS, collator akan menganggap EOS ini padding dan memberinya label -100, sehingga sinyal berhenti hilang.

In [15]:
def sft_example(country, capital):
    """Prompt + jawaban dalam satu urutan; -100 menutupi prompt, EOS ikut dinilai agar model belajar berhenti."""
    prompt_ids = tokenizer(query(country))["input_ids"]
    answer_ids = tokenizer(f" {capital}")["input_ids"] + [tokenizer.eos_token_id]
    return prompt_ids + answer_ids, [-100] * len(prompt_ids) + answer_ids


def sft_batch(pairs):
    examples = [sft_example(c, k) for c, k in pairs]
    width = max(len(ids) for ids, _ in examples)
    pad = tokenizer.eos_token_id
    batch = {
        "input_ids": torch.tensor([ids + [pad] * (width - len(ids)) for ids, _ in examples]),
        "attention_mask": torch.tensor([[1] * len(ids) + [0] * (width - len(ids)) for ids, _ in examples]),
        "labels": torch.tensor([lab + [-100] * (width - len(lab)) for _, lab in examples]),
    }
    assert (batch["labels"] != -100).any(dim=1).all(), "contoh tanpa token target akan membuat loss nan"
    return batch


ids, lab = sft_example("Germany", "Berlin")
print("token :", tokenizer.convert_ids_to_tokens(ids))
print("label :", ["-100" if t == -100 else tokenizer.convert_ids_to_tokens(t) for t in lab])

reset_gpt2()
assert n_trainable(model) == N_TOTAL                   # SFT melatih seluruh model
gen = torch.Generator().manual_seed(0)
batches = []
for _ in range(CFG["sft_epochs"]):
    order = torch.randperm(len(TRAIN_PAIRS), generator=gen).tolist()
    batches += [sft_batch([TRAIN_PAIRS[i] for i in order[b:b + 4]]) for b in range(0, len(order), 4)]
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.0)
torch.manual_seed(0)
stats = train_on(model, optimizer, batches, new_scaler())
del optimizer

sft_hits, sft_ans = accuracy(HELD_OUT)
seen_hits, _ = accuracy(TRAIN_PAIRS)
print(f"SFT {len(batches)} step (batch 4, lr 1e-4) | step dilewati GradScaler: {stats['skipped']}")
print(f"held-out tanpa contoh di prompt : {sft_hits}/{n} ({sft_hits / n:.0%})")
print(f"16 pasangan training            : {seen_hits}/{len(TRAIN_PAIRS)}")
print("jawaban held-out:", ", ".join(f"{c}->{a.strip()!r}" for (c, _), a in zip(HELD_OUT, sft_ans)))
reset_gpt2()
free_memory()

token : ['Input', ':', 'ĠGermany', 'Ċ', 'Output', ':', 'ĠBerlin', '<|endoftext|>']
label : ['-100', '-100', '-100', '-100', '-100', '-100', 'ĠBerlin', '<|endoftext|>']
SFT 24 step (batch 4, lr 1e-4) | step dilewati GradScaler: 6
held-out tanpa contoh di prompt : 13/16 (81%)
16 pasangan training            : 16/16
jawaban held-out: France->'Paris', Japan->'Hanoi', Italy->'Rome', Egypt->'Cairo', Canada->'Hanoi', Russia->'Moscow', China->'Hanoi', Ireland->'Dublin', Indonesia->'Jakarta', Belgium->'Brussels', Denmark->'Copenhagen', Hungary->'Budapest', Cuba->'Havana', Iran->'Tehran', Iraq->'Baghdad', Australia->'Canberra'


Baris `label` memperlihatkan format SFT secara harfiah: semua token prompt bernilai -100, sedangkan ` Berlin` dan `<|endoftext|>` dinilai.

Dua skor setelah SFT perlu dibaca berpasangan. Skor pada 16 pasangan training menunjukkan seberapa baik model menghafal yang sudah ia lihat. Skor held-out menunjukkan apakah yang dipelajari adalah *format*, karena 16 negara itu tidak pernah muncul di data SFT. Bila skor held-out tinggi tanpa contoh di prompt, SFT berhasil memindahkan perilaku tanya-jawab ke dalam bobot, sementara nama-nama ibu kotanya berasal dari pretraining. Few-shot fine-tuning mengajarkan *perilaku*, bukan *pengetahuan*. Karena itu in-context learning, yang tidak butuh training sama sekali, adalah pembanding yang serius; trade-off-nya adalah biaya training sekali versus prompt yang lebih panjang di setiap pemanggilan. Dengan hanya 16 negara held-out dan decoding greedy, selisih satu atau dua negara antar-run atau antar-GPU tidak perlu ditafsirkan.

## 6. Continual fine-tuning dan catastrophic forgetting

Model yang dilatih pada task A lalu pada task B bisa kehilangan kemampuan A. Fenomena ini disebut **catastrophic forgetting**. Buku membahas dua penangkal:

- **EWC** (*elastic weight consolidation*) menahan *parameter*. Setelah task A, ukur seberapa penting setiap parameter bagi A lewat **importance**, yaitu rata-rata kuadrat gradien loss A (pendekatan diagonal Fisher). Saat belajar B, tambahkan penalti λ · Σ importanceᵢ · (θᵢ − θ_A,ᵢ)². Parameter yang penting bagi A ditarik kembali ke nilai lamanya, sedangkan yang tidak penting bebas berubah.
- **LwF** (*learning without forgetting*, lewat distilasi) menahan *perilaku*. Model baru dipaksa meniru distribusi token model lama pada input task lama, diukur dengan KL divergence pada suhu T.

Agar banyak varian bisa dilatih dalam hitungan detik, eksperimen ini memakai GPT-2 mini dari config (2 blok, lebar 64, 366.336 parameter) dengan kosakata 4.095 token GPT-2 yang paling sering muncul di wikitext train, ditambah satu token "lainnya". **Task A** adalah wikitext dibaca normal, kiri ke kanan. **Task B** adalah teks sejenis dibaca terbalik, kanan ke kiri. Statistik B sengaja bertentangan dengan A, sehingga forgetting mudah terlihat. Kosakata dan data training diambil dari split train, sedangkan semua evaluasi memakai split validation.

Model mini ini sengaja dilatih float32 penuh di GPU. Pada ukuran sekitar 0,37 juta parameter, autocast tidak memberi keuntungan berarti, dan penalti EWC yang berupa selisih kuadrat kecil lebih aman tanpa pembulatan float16.

In [16]:
V, SEQ, EWC_BS = 4096, 64, 8


def wiki_token_ids(split, n_lines):
    texts = list(raw[split].select(range(min(n_lines, len(raw[split]))))["text"])
    ids = []
    for seq in tokenizer(texts)["input_ids"]:
        ids.extend(seq + [tokenizer.eos_token_id])
    return ids


train_ids = wiki_token_ids("train", CFG["ewc_lines"])
val_ids = wiki_token_ids("validation", CFG["ewc_val_lines"])
top = [t for t, _ in collections.Counter(train_ids).most_common(V - 1)]   # kosakata dari split train saja
remap = {t: i + 1 for i, t in enumerate(top)}                             # 0 = token di luar 4.095 tersering


def to_blocks(ids):
    x = torch.tensor([remap.get(t, 0) for t in ids])
    n_blocks = len(x) // SEQ
    return x[:n_blocks * SEQ].view(n_blocks, SEQ).to(DEVICE)


tr_blocks, va_blocks = to_blocks(train_ids), to_blocks(val_ids)
ntr, nva = CFG["ewc_train_blocks"], CFG["ewc_val_blocks"]
assert len(tr_blocks) >= 2 * ntr and len(va_blocks) >= 2 * nva and ntr % EWC_BS == 0
A_tr, B_tr = tr_blocks[:ntr], tr_blocks[ntr:2 * ntr].flip(1)          # task B: urutan token dibalik
A_va, B_va = va_blocks[:nva], va_blocks[nva:2 * nva].flip(1)


def mini_model():
    torch.manual_seed(42)
    cfg = GPT2Config(vocab_size=V, n_positions=SEQ, n_embd=64, n_layer=2, n_head=2,
                     resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=0, eos_token_id=0)
    return GPT2LMHeadModel(cfg).to(DEVICE)          # float32 penuh, di device gerbang


def lm_loss(m, x):
    return m(input_ids=x, labels=x).loss


@torch.no_grad()
def mini_eval(m, blocks):
    m.eval()
    return float(np.mean([lm_loss(m, x).item() for x in blocks.split(EWC_BS)]))


def mini_train(m, data, steps, objective=None, lr=1e-3):
    """objective(m, x, step) -> loss; bawaan: loss bahasa biasa pada batch x."""
    optimizer = torch.optim.AdamW(m.parameters(), lr=lr)
    m.train()
    for s in range(steps):
        i = (s * EWC_BS) % len(data)
        x = data[i:i + EWC_BS]
        loss = lm_loss(m, x) if objective is None else objective(m, x, s)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"loss non-finite di step {s}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)


m = mini_model()
loss_A0, loss_B0 = mini_eval(m, A_va), mini_eval(m, B_va)
mini_train(m, A_tr, CFG["ewc_steps_a"])
after_A = (mini_eval(m, A_va), mini_eval(m, B_va))
theta_A = {name: p.detach().clone() for name, p in m.named_parameters()}
sd_A = copy.deepcopy(m.state_dict())
print(f"model mini {sum(p.numel() for p in m.parameters()):,} parameter di {next(m.parameters()).device} | "
      f"blok {SEQ} token: train {ntr} per task, validation {nva} per task")
print(f"sebelum training : loss A {loss_A0:.3f} | loss B {loss_B0:.3f}  (acak: ln {V} = {math.log(V):.3f})")
print(f"setelah task A   : loss A {after_A[0]:.3f} | loss B {after_A[1]:.3f}")

model mini 366,336 parameter di cuda:0 | blok 64 token: train 400 per task, validation 64 per task
sebelum training : loss A 8.270 | loss B 8.278  (acak: ln 4096 = 8.318)
setelah task A   : loss A 5.460 | loss B 5.664


Baris pertama adalah titik nol: model acak memberi loss mendekati ln 4096 ≈ 8,3 nat untuk kedua task, karena ia belum lebih baik dari menebak seragam. Setelah task A, loss A seharusnya turun jauh. Loss B biasanya ikut turun sedikit, karena frekuensi token yang sama tetap berguna walau urutannya terbalik. Kedua angka "setelah task A" menjadi titik acuan berikutnya: berapa yang hilang dari A dan berapa yang didapat untuk B.

> **Listing buku: `ewc_loss` dan `compute_importance`**
> ```python
> loss += ewc_lambda * importance[n]
> * (p - old_model[n]).pow(2).sum()
> ...
> output = model(batch)
> ```
> Baris yang diawali `*` tanpa penyambung adalah `SyntaxError: can't use starred expression here`. Bila disambung, `.sum()` berada di posisi yang salah: hasilnya tensor seukuran parameter, bukan skalar, sehingga `loss +=` gagal (`RuntimeError: output with shape [] doesn't match the broadcast shape`). Yang dimaksud adalah `(importance[n] * (p - old[n]) ** 2).sum()`. Lalu `old_model[n]` mengindeks model seperti dict (`TypeError: 'GPT2LMHeadModel' object is not subscriptable`), padahal yang dibutuhkan adalah salinan nilai parameter lama. Terakhir, `model(batch)` mengirim seluruh batch sebagai argumen posisional pertama (`input_ids`); untuk batch berbentuk dict hasilnya `AttributeError`, dan tanpa `labels` tidak ada `output.loss`. Semuanya terverifikasi; versi yang berjalan ada di bawah.

Importance dinormalkan agar rata-ratanya 1. Nilai λ = 0,01 di buku tidak punya skala tetap, karena besar penaltinya bergantung pada besar gradien model dan datanya. Setelah normalisasi, λ bisa dibandingkan antar-run. Empat nilai λ dicoba, masing-masing mulai dari bobot yang sama persis setelah task A.

In [17]:
def compute_importance(model, data, n_batches=10):
    """Importance EWC: rata-rata kuadrat gradien loss task A per parameter (diagonal Fisher empiris)."""
    model.eval()
    importance = {name: torch.zeros_like(p) for name, p in model.named_parameters()}
    chunks = data[: n_batches * EWC_BS].split(EWC_BS)
    for x in chunks:
        model.zero_grad(set_to_none=True)
        lm_loss(model, x).backward()
        for name, p in model.named_parameters():
            if p.grad is not None:
                importance[name] += p.grad.detach() ** 2 / len(chunks)
    model.zero_grad(set_to_none=True)
    return importance


def ewc_loss(model, old_params, importance, loss, ewc_lambda):
    """Loss task baru + lambda * sum_i importance_i * (theta_i - theta_A,i)^2."""
    penalty = sum((importance[name] * (p - old_params[name]) ** 2).sum()
                  for name, p in model.named_parameters() if name in importance)
    return loss + ewc_lambda * penalty


importance = compute_importance(m, A_tr)
mean_importance = sum(v.sum() for v in importance.values()) / sum(v.numel() for v in importance.values())
importance = {name: v / mean_importance for name, v in importance.items()}     # rata-rata importance = 1
assert all(torch.isfinite(v).all() for v in importance.values())

results = [("setelah task A", *after_A)]
for lam in (0.0, 0.1, 1.0, 10.0):
    m.load_state_dict(sd_A)
    objective = None if lam == 0 else (
        lambda mm, x, s, lam=lam: ewc_loss(mm, theta_A, importance, lm_loss(mm, x), lam))
    mini_train(m, B_tr, CFG["ewc_steps_b"], objective)
    results.append(("B tanpa proteksi" if lam == 0 else f"B + EWC λ={lam:g}", mini_eval(m, A_va), mini_eval(m, B_va)))


def show(rows):
    print(f"{'tahap':<20}{'loss A':>8}{'loss B':>8}{'lupa A':>9}{'belajar B':>11}")
    for label, la, lb in rows:
        print(f"{label:<20}{la:>8.3f}{lb:>8.3f}{la - after_A[0]:>+9.3f}{after_A[1] - lb:>+11.3f}")


show(results)

tahap                 loss A  loss B   lupa A  belajar B
setelah task A         5.460   5.664   +0.000     +0.000
B tanpa proteksi       6.048   5.251   +0.588     +0.414
B + EWC λ=0.1          5.809   5.424   +0.349     +0.241
B + EWC λ=1            5.670   5.494   +0.210     +0.170
B + EWC λ=10           5.483   5.596   +0.023     +0.069


Kolom **lupa A** adalah kenaikan loss A dibanding setelah task A; makin besar, makin banyak yang terlupa. Kolom **belajar B** adalah penurunan loss B dibanding titik yang sama; makin besar, makin banyak B yang dipelajari. Baris "B tanpa proteksi" adalah catastrophic forgetting dalam angka: biasanya B dipelajari paling dalam, dan A paling banyak dilupakan.

Baris-baris EWC memetakan **trade-off stabilitas–plastisitas**. Menaikkan λ seharusnya memperkecil lupa A sekaligus memperkecil belajar B. Bila tidak ada satu baris pun yang unggul di kedua kolom sekaligus, itu bukan kegagalan EWC. Dua task yang sengaja bertentangan memperebutkan parameter yang sama, dan EWC hanya bisa menegosiasikan kompromi. Buku sendiri menyebut efektivitas continual fine-tuning bergantung pada kemiripan task. Pada dua domain teks yang searah, ruang komprominya jauh lebih lega.

> **Listing buku: `lwf_loss`**
> ```python
> old_outputs = old_model(old_data)
> new_outputs = model(old_data)
> distillation_loss = F.kl_div(F.log_softmax(new_outputs / temperature, dim=1),
>     F.softmax(old_outputs / temperature, dim=1), reduction='batchmean') * (temperature 2)
> ```
> `(temperature 2)` adalah `SyntaxError`; yang dimaksud `temperature ** 2`, faktor yang mengimbangi gradien soft target yang mengecil sekitar 1/T² (terverifikasi). Masalah lainnya terlihat dari kodenya: `compute_loss` dan `F` tidak didefinisikan di listing, output model berupa objek sehingga perlu `.logits`, dan untuk model bahasa softmax harus di dimensi kosakata (`dim=-1`), bukan `dim=1` yang merupakan dimensi posisi.

Versi di bawah mengikuti tanda tangan fungsi buku, yang memakai data task lama. Model lama (*teacher*) adalah salinan beku dari model setelah task A. Satu catatan: LwF versi aslinya mendistilasi pada input task *baru*, sehingga sama sekali tidak butuh data lama. Varian buku lebih dekat ke *rehearsal* dengan distilasi.

In [18]:
def lwf_loss(model, old_model, new_loss, old_data, temperature=2.0):
    """Distilasi: model baru meniru distribusi token model lama pada data task lama."""
    with torch.no_grad():
        old_logits = old_model(input_ids=old_data).logits
    new_logits = model(input_ids=old_data).logits
    vocab = new_logits.size(-1)
    distill = F.kl_div(F.log_softmax(new_logits.reshape(-1, vocab) / temperature, dim=-1),
                       F.softmax(old_logits.reshape(-1, vocab) / temperature, dim=-1),
                       reduction="batchmean") * temperature ** 2          # rata-rata per token
    return new_loss + distill


m.load_state_dict(sd_A)
teacher = copy.deepcopy(m).eval()
for p in teacher.parameters():
    p.requires_grad_(False)


def lwf_objective(mm, x, s):
    j = (s * EWC_BS) % len(A_tr)
    return lwf_loss(mm, teacher, lm_loss(mm, x), A_tr[j:j + EWC_BS])


mini_train(m, B_tr, CFG["ewc_steps_b"], lwf_objective)
lwf_row = ("B + LwF T=2", mini_eval(m, A_va), mini_eval(m, B_va))
assert all(math.isfinite(v) for _, la, lb in results + [lwf_row] for v in (la, lb))
show([results[1], lwf_row])
del m, teacher
free_memory()

tahap                 loss A  loss B   lupa A  belajar B
B tanpa proteksi       6.048   5.251   +0.588     +0.414
B + LwF T=2            5.673   5.176   +0.213     +0.488


LwF membayar dengan compute: setiap step menjalankan dua forward tambahan pada batch task A, satu di teacher dan satu di model yang sedang dilatih. Bandingkan barisnya dengan "B tanpa proteksi" dan dengan baris EWC sebelumnya. Bila lupa A-nya lebih kecil tetapi belajar B-nya tetap baik, menahan *perilaku* pada data lama lebih efisien daripada menahan parameter untuk pasangan task ini. Bila hasilnya mirip EWC dengan λ besar, kedua penangkal sedang menghadapi konflik yang sama.

Buku juga menyebut **GEM** (*gradient episodic memory*), yang memproyeksikan gradien task baru agar tidak menaikkan loss pada contoh-contoh lama yang disimpan. GEM tidak diimplementasikan di notebook ini.

## Ringkasan

- **Fine-tuning dasar.** `Trainer` butuh `labels`. Untuk model kausal, `DataCollatorForLanguageModeling(mlm=False)` menyalin `input_ids` dan memberi -100 pada padding. Perubahan validation loss Δ nat berarti perplexity dikali e^Δ, jadi penurunan kecil dalam nat tetap bisa berarti perplexity turun belasan persen.
- **Freezing.** Jumlah parameter yang dilatih turun pasti, dari 124,4M ke 42,5M (top-6) atau 14,2M (top-2), dan memori optimizer ikut turun sekitar 12 byte per parameter yang dibekukan. Apakah penghematan waktu dan memori sepadan dengan adaptasi yang lebih dangkal, jawabannya ada di kolom turun dan turun/detik pada run Anda. `lm_head` yang tied ikut beku bersama embedding.
- **Gradual unfreezing.** Mencairkan layer tidak ada artinya bila optimizer tidak diberi tahu. `add_param_group` memasukkan parameter baru tanpa membuang state Adam yang lama.
- **Schedule.** Scheduler hanya bekerja bila benar-benar dipakai: serahkan lewat `optimizers=(...)` atau pilih dengan `lr_scheduler_type`. Cosine mempertahankan learning rate lebih tinggi di awal peluruhan dan lebih rendah di akhir.
- **Domain.** Packing menggantikan `TextDataset` tanpa padding dan tanpa pemotongan. Evaluasi dua arah, domain dan umum, memperlihatkan adaptasi beserta efek sampingnya.
- **Few-shot.** Contoh di prompt dan SFT 16 contoh sama-sama mengajarkan format; pengetahuannya berasal dari pretraining. Format SFT yang benar menaruh jawaban di dalam `input_ids` dan menutupi prompt dengan -100.
- **Continual learning.** Forgetting diukur sebagai kenaikan loss task lama. EWC dan LwF menggeser trade-off stabilitas–plastisitas, tetapi tidak menghapusnya untuk task yang saling bertentangan.

Bab berikutnya membahas **model pruning**: memangkas bagian model yang redundan sambil mempertahankan fungsinya.